# Tutorial 5: Neural Architecture Search (NAS) with Mase and Optuna

## Contents

- [1. Defining the Search Space](#1-defining-the-search-space)
- [2. Writing a Model Constructor](#2-writing-a-model-constructor)
- [3. Defining the Objective Function](#3-defining-the-objective-function)
- [4. Launching the Search](#4-launching-the-search)
- [Deploying the Optimized Model with CompressionPipeline](#deploying-the-optimized-model-with-compressionpipeline)

---

In this tutorial, we'll see how Mase can be integrated with Optuna, the popular hyperparameter optimization framework, to search for a Bert model optimized for sequence classification on the IMDb dataset. We'll take the Optuna-generated model and import it into Mase, then run the CompressionPipeline to prepare the model for edge deployment by quantizing and pruning its weights.

As we'll see, running Architecture Search with Mase/Optuna involves the following steps:

1. **Define the search space**: this is a dictionary containing the range of values for each parameter at each layer in the model.
2. **Write the model constructor**: this is a function which uses Optuna utilities to sample a model from the search space, and constructs the model using transformers `from_config` class method.
3. **Write the objective function**: this function calls on the model constructor defined in Step 2 and defines the training/evaluation setup for each search iteration.
4. **Go!** Choose an Optuna sampler, create a study and launch the search.

```python
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"
```

First, fetch the dataset using the `get_tokenized_dataset` utility.

```python
from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)
```

## 1. Defining the Search Space

We'll start by defining a search space, i.e. enumerating the possible combinations of hyperparameters that Optuna can choose during search. We'll explore the following range of values for the model's hidden size, intermediate size, number of layers and number of heads, inspired by the [NAS-BERT paper](https://arxiv.org/abs/2105.14444).

```python
import torch.nn as nn
from chop.nn.modules import Identity

search_space = {
    "num_layers": [2, 4, 8],
    "num_heads": [2, 4, 8, 16],
    "hidden_size": [128, 192, 256, 384, 512],
    "intermediate_size": [512, 768, 1024, 1536, 2048],
    "linear_layer_choices": [
        nn.Linear,
        Identity,
    ],
}
```

## 2. Writing a Model Constructor

We define the following function, which will get called in each iteration of the search process. The function is passed the `trial` argument, which is an Optuna object that comes with many functionalities - see the [Trial documentation](https://optuna.readthedocs.io/en/stable/reference/trial.html) for more details. Here, we use the `trial.suggest_int` and `trial.suggest_categorical` functions to trigger the chosen sampler to choose parameter choices and layer types. The suggested integer is the index into the search space for each parameter, which we defined in the previous cell.

```python
from transformers import AutoConfig, AutoModelForSequenceClassification
from chop.tools.utils import deepsetattr


def construct_model(trial):
    config = AutoConfig.from_pretrained(checkpoint)

    # Update the paramaters in the config
    for param in [
        "num_layers",
        "num_heads",
        "hidden_size",
        "intermediate_size",
    ]:
        chosen_idx = trial.suggest_int(param, 0, len(search_space[param]) - 1)
        setattr(config, param, search_space[param][chosen_idx])

    trial_model = AutoModelForSequenceClassification.from_config(config)

    for name, layer in trial_model.named_modules():
        if isinstance(layer, nn.Linear) and layer.in_features == layer.out_features:
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            if new_layer_cls == nn.Linear:
                continue
            elif new_layer_cls == Identity:
                new_layer = Identity()
                deepsetattr(trial_model, name, new_layer)
            else:
                raise ValueError(f"Unknown layer type: {new_layer_cls}")

    return trial_model
```

## 3. Defining the Objective Function

Next, we define the objective function for the search, which gets called on each trial. In each trial, we create a new model instance with chosen hyperparameters according to the defined sampler. We then use the `get_trainer` utility in Mase to run a training loop on the IMDb dataset for a number of epochs. Finally, we use `evaluate` to report back the classification accuracy on the test split.

```python
from chop.tools import get_trainer


def objective(trial):

    # Define the model
    model = construct_model(trial)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    # Set the model as an attribute so we can fetch it later
    trial.set_user_attr("model", model)

    return eval_results["eval_accuracy"]
```

## 4. Launching the Search

Optuna provides a number of samplers, for example:

- **GridSampler**: iterates through every possible combination of hyperparameters in the search space
- **RandomSampler**: chooses a random combination of hyperparameters in each iteration
- **TPESampler**: uses Tree-structured Parzen Estimator algorithm to choose hyperparameter values.

You can define the chosen sampler by simply importing from `optuna.samplers` as below.

```python
from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = RandomSampler()
```

With all the pieces in place, we can launch the search as follows. The number of trials is set to 1 so you can go get a coffee for 10 minutes, then proceed with the tutorial. However, this will essentially be a random model - for better results, set this to 100 and leave it running overnight!

```python
import optuna

study = optuna.create_study(
    direction="maximize",
    study_name="bert-tiny-nas-study",
    sampler=sampler,
)

study.optimize(
    objective,
    n_trials=1,
    timeout=60 * 60 * 24,
)
```

Fetch the model associated with the best trial as follows, and export to be used in future tutorials. In Tutorial 6, we'll see how to run mixed-precision quantization search on top of the model we've just found through NAS to further find the optimal quantization mapping.

```python
from pathlib import Path
import dill

model = study.best_trial.user_attrs["model"].cpu()

with open(f"{Path.home()}/tutorial_5_best_model.pkl", "wb") as f:
    dill.dump(model, f)
```

## Deploying the Optimized Model with CompressionPipeline

Now, we can run the CompressionPipeline in Mase to run uniform quantization and pruning over the searched model.

```python
from chop.pipelines import CompressionPipeline
from chop import MaseGraph

mg = MaseGraph(model)
pipe = CompressionPipeline()

quantization_config = {
    "by": "type",
    "default": {
        "config": {
            "name": None,
        }
    },
    "linear": {
        "config": {
            "name": "integer",
            # data
            "data_in_width": 8,
            "data_in_frac_width": 4,
            # weight
            "weight_width": 8,
            "weight_frac_width": 4,
            # bias
            "bias_width": 8,
            "bias_frac_width": 4,
        }
    },
}

pruning_config = {
    "weight": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
    "activation": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
}

mg, _ = pipe(
    mg,
    pass_args={
        "quantize_transform_pass": quantization_config,
        "prune_transform_pass": pruning_config,
    },
)
```

Finally, export the MaseGraph for the compressed checkpoint to be used in future tutorials for hardware generation and distributed deployment.

```python
mg.export(f"{Path.home()}/tutorial_5_nas_compressed", save_format="state_dict")
```

---

*By DeepWok*

*© Copyright 2023, DeepWok.*

---
---

In [ ]:
import sys
sys.version

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version
!git clone https://github.com/DeepWok/mase.git
%cd mase
!uv sync
import glob, site, sys

venv_site = glob.glob("/content/mase/.venv/lib/python*/site-packages")[0]
site.addsitedir(venv_site)
sys.path.insert(0, venv_site)

print("Using:", venv_site)

In [ ]:

!uv run pytest test/ir/graph/test_create_masegraph.py
!uv run python -c "import chop; import transformers; print('Environment ready!')"

1. Tutorial 5 shows how to use random search to find the optimal configuration of hyperparameters and layer choices for the Bert model.

    (a) Now, explore using the GridSampler and TPESampler in Optuna.

    (b) Plot a figure that has the number of trials on the x axis, and the maximum achieved accuracy up to that point on the y axis. Plot one curve for each sampler to compare their performance.

2. In Tutorial 5, NAS is used to find an optimal configuration of hyperparameters, then we use the CompressionPipeline in Mase to quantize and prune the model after search is finished. However, the final compressed model may not be optimal, since different model architectures may have different sensitivities to quantization and pruning. Ideally, we want to run a compression-aware search flow, where the quantization and pruning is considered in each trial.

    (a) In the objective function, after the model is constructed and trained for some iterations, call the CompressionPipeline to quantize and prune the model, then continue training for a few more epochs. Use the sampler that yielded the best results in Task 1 to run the compression-aware search. The objective function should return the final accuracy of the model after compression. Consider also the case where final training is performed after quantization/pruning.

    (b) Plot a new figure that has the number of trials on the x axis, and the maximum achieved accuracy up to that point on the y axis. There should be three curves: 1. the best performance from Task 1 (without compression), compression-aware search without post-compression training, and compression-aware search with post-compression training.

In [ ]:
import torch.nn as nn
from transformers.trainer import Trainer
from chop.nn.modules import Identity
from chop.tools.utils import deepsetattr
from transformers import AutoConfig, AutoModelForSequenceClassification
from chop.tools import get_trainer
import optuna
from optuna.trial import TrialState
import torch
from chop.pipelines import CompressionPipeline
from chop import MaseGraph
from chop.tools import get_tokenized_dataset
from typing import Optional
import matplotlib.pyplot as plt
from optuna.samplers import GridSampler, TPESampler
from optuna.trial import TrialState
import matplotlib.pyplot as plt
from pathlib import Path

search_space = {
    "num_layers": [2, 4, 8],
    "num_heads": [2, 4, 8, 16], 
    "hidden_size": [128, 192, 256, 384, 512],
    "intermediate_size": [512, 768, 1024, 1536, 2048],

    # Global layer policy (GridSampler-friendly)
    "linear_layer_policy": ["keep_linear", "use_identity"],
}

ARTIFACT_ROOT = Path("optuna_artifacts")

def save_model_for_trial(trial, model, tag="final"):
    out_dir = ARTIFACT_ROOT / trial.study.study_name
    out_dir.mkdir(parents=True, exist_ok=True)

    model_path = out_dir / f"trial_{trial.number}_{tag}.pt"

    model_cpu = model.to("cpu")
    torch.save(model_cpu, model_path)

    # store only strings in user_attrs (works with any Optuna storage)
    trial.set_user_attr("model_path", str(model_path))
    return str(model_path)

def cap_trainer_steps(trainer : Trainer, max_steps: int):
    trainer.args.max_steps = max_steps
    trainer.args.save_strategy = "no"
    trainer.args.evaluation_strategy = "no"
    trainer.args.logging_steps = max(1, max_steps // 5)
    return trainer

def construct_model(trial):
    config = AutoConfig.from_pretrained(checkpoint)

    for param in [
        "num_layers",
        "num_heads",
        "hidden_size",
        "intermediate_size",
    ]:
        chosen = trial.suggest_categorical(param, search_space[param])
        setattr(config, param, chosen)

    model = AutoModelForSequenceClassification.from_config(config)

    # One global layer choice to keep the search space fixed for GridSampler
    policy = trial.suggest_categorical(
        "linear_layer_policy", search_space["linear_layer_policy"]
    )

    if policy == "use_identity":
        # Replace only square Linear layers (in_features == out_features)
        for name, layer in model.named_modules():
            if isinstance(layer, nn.Linear) and layer.in_features == layer.out_features:
                deepsetattr(model, name, Identity())

    return model

def base_objective(trial) :
    model = construct_model(trial)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    save_model_for_trial(trial, trainer.model, tag="base_final")

    
    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return eval_results["eval_accuracy"]

def compression_objective(trial):

    flow = trial.study.user_attrs.get("flow", {})
    run_pretrain     = bool(flow.get("run_pretrain", True))
    run_compression  = bool(flow.get("run_compression", True))
    run_posttrain    = bool(flow.get("run_posttrain", True))
    pre_max_steps    = int(flow.get("pre_max_steps", 50))
    post_max_steps   = int(flow.get("post_max_steps", 100))

    # Compression configs
    quant_cfg = trial.study.user_attrs.get("quantization_config", None)
    prune_cfg = trial.study.user_attrs.get("pruning_config", None)

    model = construct_model(trial)

    model_for_compression = model

    pre_trainer = None
    if run_pretrain:
        pre_trainer = get_trainer(
            model=model,
            tokenized_dataset=dataset,
            tokenizer=tokenizer,
            evaluate_metric="accuracy",
            num_train_epochs=1,
        )
        cap_trainer_steps(pre_trainer, pre_max_steps)
        pre_trainer.train()
        model_for_compression = pre_trainer.model  # trained weights

    final_model = model_for_compression
    mg = None
    if run_compression:
        pipe = CompressionPipeline()
        mg = MaseGraph(model_for_compression)
        mg, _ = pipe(
            mg,
            pass_args={
                "quantize_transform_pass": quant_cfg,
                "prune_transform_pass": prune_cfg,
            },
        )
        final_model = mg.model

    # Rebuild Trainer after compression because optimizer/scheduler must match new params.
    post_trainer = get_trainer(
        model=final_model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    if run_posttrain:
        cap_trainer_steps(post_trainer, post_max_steps)
        post_trainer.train()

    eval_results = post_trainer.evaluate()
    acc = eval_results["eval_accuracy"]

    save_model_for_trial(trial, post_trainer.model, tag="compression_aware_final")

    # keep GPU memory from snowballing between trials
    del post_trainer, pre_trainer, mg, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return acc

def run_search(
    objective,
    sampler,
    name,
    flow : dict,
    direction = "maximize",
    quantization_config : Optional[dict] = None,
    pruning_config : Optional[dict] = None,
    n_trials=10,
    timeout=60*60,
):
    study = optuna.create_study(
        direction=direction,
        study_name=name,
        sampler=sampler,
    )

    study.set_user_attr("quantization_config", quantization_config)
    study.set_user_attr("pruning_config", pruning_config)
    study.set_user_attr("flow", flow)

    study.optimize(objective, n_trials=n_trials, timeout=timeout)
    return study

def cumulative_best_values(study: optuna.Study):
    vals = [
        t.value for t in study.trials
        if t.state == TrialState.COMPLETE and t.value is not None
    ]
    best = []
    cur = float("-inf")
    for v in vals:
        cur = max(cur, v)
        best.append(cur)
    return best


def best_so_far_curve(study: optuna.Study):
    """Return max-so-far curve over completed trials."""
    best = float("-inf")
    curve = []
    for t in study.trials:
        if t.state == TrialState.COMPLETE and t.value is not None:
            best = max(best, t.value)
            curve.append(best)
    return curve


def plot_curves(curves, title):
    plt.figure(figsize=(9, 5))
    for label, y in curves.items():
        x = list(range(1, len(y) + 1))
        plt.plot(x, y, label=label)
    plt.xlabel("Number of trials")
    plt.ylabel("Best accuracy so far")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


## Task 1

In [ ]:
import torch.nn as nn
import optuna
from optuna.samplers import GridSampler, TPESampler
from optuna.trial import TrialState
import matplotlib.pyplot as plt
from typing import Optional


samplers = [GridSampler, TPESampler]
SEED = 42

grid_search_space = {
    "num_hidden_layers": search_space["num_layers"],
    "num_attention_heads": search_space["num_heads"],
    "hidden_size": search_space["hidden_size"],
    "intermediate_size": search_space["intermediate_size"],
    "linear_layer_policy": search_space["linear_layer_policy"],
}

# Total combos for Grid 
total_grid_trials = 1
for v in grid_search_space.values():
    total_grid_trials *= len(v)

grid_sampler = GridSampler(grid_search_space, seed=SEED)
tpe_sampler = TPESampler(seed=SEED, multivariate=True)


# Trials to compare (same budget = fair comparison)
BUDGET = min(10, total_grid_trials) 
timeout = 60 * 60 * 34

grid_study = run_search(
    base_objective, 
    grid_sampler, 
    name="grid", 
    flow = {
        "run_pretrain": False,
        "run_compression": False,
        "run_posttrain": True,
        "pre_max_steps": 50,
    },
    n_trials=BUDGET, 
    timeout=timeout,
)

tpe_study  = run_search(
    base_objective, 
    tpe_sampler,  
    name="tpe",  
    flow = {
        "run_pretrain": False,
        "run_compression": False,
        "run_posttrain": True,
        "pre_max_steps": 50,
    },
    n_trials=BUDGET, 
    timeout=timeout
)

plot_curves(
    {
        "GridSampler": best_so_far_curve(grid_study),
        "TPESampler": best_so_far_curve(tpe_study),
    },
    title="Task 1: Sampler comparison (NAS, no compression)",
)

## Task 2

In [ ]:
from chop.pipelines import CompressionPipeline
from chop import MaseGraph


quantization_config = {
    "by": "type",
    "default": {
        "config": {
            "name": None,
        }
    },
    "linear": {
        "config": {
            "name": "integer",
            # data
            "data_in_width": 8,
            "data_in_frac_width": 4,
            # weight
            "weight_width": 8,
            "weight_frac_width": 4,
            # bias
            "bias_width": 8,
            "bias_frac_width": 4,
        }
    },
}

pruning_config = {
    "weight": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
    "activation": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
}

best_sampler = None
best_sampler_name = None

# 1) Task 1 baseline (no compression)
study_baseline = run_search(
    objective = compression_objective,
    sampler=best_sampler,
    direction="maximize",
    name="baseline_no_compression",
    quantization_config=quantization_config,
    pruning_config=pruning_config,
    flow={
        "run_pretrain": True,
        "run_compression": False,
        "run_posttrain": False,
        "pre_max_steps": 50,
    },
    n_trials=10,
)

# Compression-aware, no post-training
study_ca_no_post = run_search(
    objective = compression_objective,
    sampler=best_sampler,
    direction="maximize",
    name="ca_no_post",
    quantization_config=quantization_config,
    pruning_config=pruning_config,
    flow={
        "run_pretrain": True,
        "run_compression": True,
        "run_posttrain": False,
        "pre_max_steps": 50,
    },
    n_trials=10,
)

# Compression-aware, with post-training
study_ca_with_post = run_search(
    objective = compression_objective,
    sampler=best_sampler,
    direction="maximize",
    name="ca_with_post",
    quantization_config=quantization_config,
    pruning_config=pruning_config,
    flow={
        "run_pretrain": True,
        "run_compression": True,
        "run_posttrain": True,
        "pre_max_steps": 50,
        "post_max_steps": 100,
    },
    n_trials=10,
)

plot_curves(
    {
        f"Task 1 best (no compression) [{best_sampler_name}]": best_so_far_curve(study_baseline),
        "Compression-aware (no post-train)": best_so_far_curve(study_ca_no_post),
        "Compression-aware (+ post-train)": best_so_far_curve(study_ca_with_post),
    },
    title="Task 2: Compression-aware NAS vs baseline",
)